# Part 1 QC — About Us Page Scrape Quality Control

This notebook inspects the Part 1 dataset for coverage gaps, short texts, and theme score distributions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

df = pd.read_csv('../data/processed/part1_dataset.csv')
print(df.shape)
df.head()

In [ ]:
# Coverage by year and sector
coverage = df.groupby(['sector', 'year'])['selection_status'].value_counts().unstack(fill_value=0)
coverage

In [ ]:
# Word count distribution
df['word_count'] = df['page_text_clean'].fillna('').apply(lambda x: len(str(x).split()))
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df[df['word_count'] > 0]['word_count'], bins=40, ax=ax)
ax.set_title('Distribution of Page Text Word Counts (Part 1)')
ax.set_xlabel('Word Count')
plt.show()

In [ ]:
# Theme score heatmap across companies
theme_cols = [c for c in df.columns if c.startswith('theme_') and c != 'theme_categories']
theme_means = df.groupby('ticker')[theme_cols].mean()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(theme_means, cmap='YlOrRd', vmin=0, vmax=3, annot=False, ax=ax)
ax.set_title('Mean Theme Scores by Company (About Us Pages)')
plt.tight_layout()
plt.show()

In [ ]:
# Flag missing and short snapshots
missing = df[df['selection_status'] == 'missing']
short = df[df['word_count'] < 30]
print(f'Missing snapshots: {len(missing)}')
print(f'Short texts (<30 words): {len(short)}')
missing[['ticker', 'year', 'selection_status']].sort_values(['ticker', 'year'])